In [31]:
import pandas as pd

In [32]:
df = pd.read_parquet("../data/processed/dvf_77_2025.parquet")
print(df.shape)
df.dtypes

(64913, 40)


id_mutation                         str
date_mutation                       str
numero_disposition                int64
nature_mutation                     str
valeur_fonciere                 float64
adresse_numero                  float64
adresse_suffixe                     str
adresse_nom_voie                    str
adresse_code_voie                   str
code_postal                         str
code_commune                        str
nom_commune                         str
code_departement                    str
ancien_code_commune                 str
ancien_nom_commune                  str
id_parcelle                         str
ancien_id_parcelle                  str
numero_volume                       str
lot1_numero                         str
lot1_surface_carrez             float64
lot2_numero                         str
lot2_surface_carrez             float64
lot3_numero                         str
lot3_surface_carrez             float64
lot4_numero                         str


In [33]:
valeurs_uniques = df['nature_mutation'].unique()
print(valeurs_uniques)

<ArrowStringArray>
[                             'Vente', 'Vente en l'état futur d'achèvement',
              'Vente terrain à bâtir',                            'Echange',
                       'Adjudication',                      'Expropriation']
Length: 6, dtype: str


In [34]:
nbr_vente = df['nature_mutation'].value_counts()
print(nbr_vente)

nature_mutation
Vente                                 58727
Vente en l'état futur d'achèvement     5117
Echange                                 693
Vente terrain à bâtir                   187
Adjudication                            175
Expropriation                            14
Name: count, dtype: int64


In [35]:
# Groupement par nature de mutation et comptage des id_mutation uniques
resultat = df.groupby('nature_mutation')['id_mutation'].nunique()

print(resultat)

nature_mutation
Adjudication                             62
Echange                                 407
Expropriation                            14
Vente                                 21317
Vente en l'état futur d'achèvement     1677
Vente terrain à bâtir                    53
Name: id_mutation, dtype: int64


In [36]:
df_vente = df[df['nature_mutation'] == 'Vente']
print(df_vente.shape)

(58727, 40)


In [37]:
# Groupement par nature de mutation et comptage des id_mutation uniques
resultat = df_vente.groupby('nature_mutation')['id_mutation'].nunique()

print(resultat)

nature_mutation
Vente    21317
Name: id_mutation, dtype: int64


In [38]:
print(df.groupby("id_mutation")["nature_mutation"].nunique().max())

2


In [39]:
natures_par_vente = df.groupby("id_mutation")["nature_mutation"].nunique()
natures_par_vente[natures_par_vente > 1]

id_mutation
2025-1050579    2
Name: nature_mutation, dtype: int64

In [40]:
colonnes = ["id_mutation", "date_mutation", "nature_mutation", "valeur_fonciere",
            "nom_commune", "type_local", "surface_reelle_bati", "nature_culture"]
df[df["id_mutation"] == "2025-1050579"][colonnes]

,id_mutation,date_mutation,nature_mutation,valeur_fonciere,nom_commune,type_local,surface_reelle_bati,nature_culture
25823,2025-1050579,2025-12-23,Echange,1.0,Pontault Combault,NaN,NaN,sols
25824,2025-1050579,2025-12-23,Echange,1.0,Pontault Combault,NaN,NaN,sols
25825,2025-1050579,2025-12-23,Vente,1.0,Melun,NaN,NaN,NaN
25826,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25827,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25828,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25829,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25830,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25831,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN
25832,2025-1050579,2025-12-23,Vente,1.0,Melun,Dépendance,0.0,NaN


In [41]:
# 1. Identifiants des ventes qui portent plusieurs natures
ids_a_exclure = natures_par_vente[natures_par_vente > 1].index

# 2. DataFrame sans ces ventes
df_coherent = df[~df["id_mutation"].isin(ids_a_exclure)]

# 3. Vérifications
print(df_coherent.groupby("id_mutation")["nature_mutation"].nunique().max())
print(len(df) - len(df_coherent))



1
13


In [42]:
df_vente = df_coherent[df_coherent['nature_mutation'] == 'Vente'].copy()


In [43]:
print(df_vente.shape)

(58716, 40)


In [44]:
resultat = df_vente.groupby('nature_mutation')['id_mutation'].nunique()
print(resultat)

nature_mutation
Vente    21316
Name: id_mutation, dtype: int64


In [45]:
df_vente["est_logement"] = df_vente["type_local"].isin(["Maison", "Appartement"])
logements_par_vente = df_vente.groupby("id_mutation")["est_logement"].sum()
logements_par_vente.head()

id_mutation
2025-1040912    1
2025-1040913    1
2025-1040914    1
2025-1040915    1
2025-1040916    1
Name: est_logement, dtype: int64

In [46]:
logements_par_vente.value_counts().sort_index()

est_logement
0       4892
1      13950
2       2163
3        135
4         84
5         17
6         31
7         11
8          4
9          6
10         1
11         1
12         3
14         1
16         1
18         1
19         1
28         1
36         1
43         1
44         1
46         1
56         1
65         1
73         1
76         1
88         1
97         1
137        1
146        1
258        1
Name: count, dtype: int64

In [47]:
logement_258  = logements_par_vente[logements_par_vente == 258]
vente_258 = df_vente[df_vente["id_mutation"].isin(logement_258.index)]

In [48]:
print(vente_258["valeur_fonciere"].unique())
print(vente_258["nom_commune"].unique())
print(vente_258["type_local"].value_counts(dropna=False))

[579123.36]
<ArrowStringArray>
['Noisiel']
Length: 1, dtype: str
type_local
Appartement                                 258
Dépendance                                  102
Local industriel. commercial ou assimilé      2
NaN                                           1
Name: count, dtype: int64


In [49]:
logement_1 = logements_par_vente[logements_par_vente == 1]

vente_1 = df_vente[df_vente["id_mutation"].isin(logement_1.index)]

In [50]:
print(vente_1["type_local"].value_counts(dropna=False))

print(vente_1.head())

type_local
Dépendance                                  11810
Maison                                       7881
Appartement                                  6069
NaN                                          5086
Local industriel. commercial ou assimilé      268
Name: count, dtype: int64
    id_mutation date_mutation  numero_disposition nature_mutation  \
0  2025-1040912    2025-01-06                   1           Vente   
1  2025-1040912    2025-01-06                   1           Vente   
2  2025-1040912    2025-01-06                   1           Vente   
3  2025-1040913    2025-01-03                   1           Vente   
4  2025-1040913    2025-01-03                   1           Vente   

   valeur_fonciere  adresse_numero adresse_suffixe     adresse_nom_voie  \
0         121000.0            31.0             NaN       RUE SERMONOISE   
1         121000.0            31.0             NaN       RUE SERMONOISE   
2         121000.0            31.0             NaN       RUE SERMONOISE  

In [51]:
vente_1["a_local"] = vente_1["type_local"].isin(["Local industriel. commercial ou assimilé"])
vente_local = vente_1.groupby("id_mutation")["a_local"].any().sum()
print(vente_local)

199


In [58]:
vente_local = vente_1.groupby("id_mutation")["a_local"].any()
ids_avec_local = vente_local[vente_local].index

df_regle1 = vente_1[~vente_1["id_mutation"].isin(ids_avec_local)].copy()
print(df_regle1["id_mutation"].nunique())

13751


In [61]:
verif_logement = df_regle1["type_local"].value_counts(dropna= False)
print(verif_logement)

type_local
Dépendance     11636
Maison          7751
Appartement     6000
NaN             4803
Name: count, dtype: int64


In [67]:
df_regle1["est_logement"] = df_regle1["type_local"].isin(["Maison","Appartement"])

verif_1_logement = df_regle1.groupby("id_mutation")["est_logement"].sum()

print(verif_1_logement.max())
print(verif_1_logement.min())

1
1
